In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../../Fase02/data/manutencao/dados_manutencao.csv")

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Data de Produção Acumulada   | Cod. Ordem   | Cod Recurso   | Cod Produto   | Qt. Total Acumulada Produzida até a data específica   | Qt. Acumulada Refugada até a data específica   | Qtd. Acumulada total Retrabalhada até a data específica   | Fator Un.   | Cód. Un.   | Descrição da massa (Composto)   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   |
|:-----------------------------|:-------------|:--------------|:--------------|:------------------------------------------------------|:-----------------------------------------------|:----------------------------------------------------------|:------------|:-----------|:--------------------------------|:-----------------------------------|:---------------------------------|
| 2023-09-08                   | 4458603      | IJ-117        | SA02961       | 87                                                    | 98                                             | 14                                                        |

## Pré-processamento dos Dados

Vou agora remover as colunas `Data de Produção Acumulada`, `Cod. Ordem`, `Cod Recurso`, `Fator Un.`, `Cód. Un.`, e `Descrição da massa (Composto)`, pois elas não são relevantes para a previsão do `Tempo Restante para Manutenção`.

As colunas `Qt. Total Acumulada Produzida até a data específica`, `Qt. Acumulada Refugada até a data específica`, e `Qtd. Acumulada total Retrabalhada até a data específica` representam quantidades acumuladas até uma data específica, então vou renomeá-las para `Qtd_Produzida`, `Qtd_Refugada` e `Qtd_Retrabalhada`, respectivamente. 

```python
# ... (código para carregar o DataFrame df) ...

# Remover colunas irrelevantes
df = df.drop(['Data de Produção Acumulada', 'Cod. Ordem', 'Cod Recurso', 'Fator Un.', 'Cód. Un.', 'Descrição da massa (Composto)'], axis=1)

# Renomear colunas
df = df.rename(columns={
    'Qt. Total Acumulada Produzida até a data específica': 'Qtd_Produzida',
    'Qt. Acumulada Refugada até a data específica': 'Qtd_Refugada',
    'Qtd. Acumulada total Retrabalhada até a data específica': 'Qtd_Retrabalhada'
})

# ... (código para continuar o pré-processamento) ...

In [2]:
# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
    }
)

# Drop unnecessary columns
df = df.drop(
    [
        "Data de Produção Acumulada",
        "Cod. Ordem",
        "Cod Recurso",
        "Fator Un.",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Cod Produto   | Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   |
|:--------------|:----------------|:---------------|:-------------------|:-----------------------------------|:---------------------------------|
| SA02961       | 87              | 98             | 14                 | 0.909355                           | -157                             |
| SA05780       | 819             | 1              | 8                  | 0.796544                           | -195                             |
| SA05780       | 84              | 74             | 4                  | 0.686085                           | -153                             |
| SA05780       | 868             | 19             | 47                 | 1.22212                            | -329                             |
| SA02961       | 95              | 69             | 25                 | 0.753044                           | -184         

# Previsão de Tempo Restante para Manutenção

Vou usar as colunas `Qtd_Produzida`, `Qtd_Refugada`, `Qtd_Retrabalhada`, `Consumo total de Massa Acumulada`, e `Cod Produto` para prever a coluna `Tempo Restante para Manutenção` usando os seguintes algoritmos de regressão:

* Support Vector Regressor (SVR)
* XGBoost Regressor
* Random Forest Regressor
* Decision Tree Regressor

Vou avaliar o desempenho de cada modelo usando as métricas:

* **Mean Squared Error (MSE)**
* **R-squared**

In [4]:
# One-hot encode categorical variables
df = pd.get_dummies(df, columns=["Cod Produto"])

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   | Cod Produto_SA02004   | Cod Produto_SA02961   | Cod Produto_SA05780   |
|:----------------|:---------------|:-------------------|:-----------------------------------|:---------------------------------|:----------------------|:----------------------|:----------------------|
| 87              | 98             | 14                 | 0.909355                           | -157                             | False                 | True                  | False                 |
| 819             | 1              | 8                  | 0.796544                           | -195                             | False                 | False                 | True                  |
| 84              | 74             | 4                  | 0.686085                           | -153                             | False                 | False                 | True          

In [5]:
# Dividir os dados em conjuntos de treinamento e teste e, em seguida, construir e avaliar modelos de aprendizado de máquina para prever o Tempo Restante para Manutenção.

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Split into features and target
X = df.drop("Tempo Restante para Manutenção", axis=1)
y = df["Tempo Restante para Manutenção"]

In [6]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and evaluate models
models = {
    "SVR": SVR(kernel="rbf"),
    "XGBoost": xgb.XGBRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100),
    "Decision Tree": DecisionTreeRegressor(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name}:")
    print(f"  Mean Squared Error: {mse:.2f}")
    print(f"  R-squared: {r2:.2f}")

SVR:
  Mean Squared Error: 10939.53
  R-squared: 0.00
XGBoost:
  Mean Squared Error: 12680.60
  R-squared: -0.16
Random Forest:
  Mean Squared Error: 11489.83
  R-squared: -0.05
Decision Tree:
  Mean Squared Error: 18737.38
  R-squared: -0.71


# Previsão de Tempo Restante para Manutenção

Vou usar as colunas `Qtd_Produzida`, `Qtd_Refugada`, `Qtd_Retrabalhada`, `Consumo total de Massa Acumulada`, e `Cod Produto` para prever a coluna `Tempo Restante para Manutenção` usando os seguintes algoritmos de regressão:

* Support Vector Regressor (SVR)
* Gradient Boosting Regressor
* Random Forest Regressor
* Decision Tree Regressor

Vou avaliar o desempenho de cada modelo usando as métricas:

* **Mean Squared Error (MSE)**
* **R-squared**


```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Carregar os dados (substitua 'seu_arquivo.csv' pelo nome do seu arquivo)
df = pd.read_csv('seu_arquivo.csv')

# ... (código para pré-processar os dados, se necessário) ...

# Defina as features (X) e o target (y)
X = df[['Qtd_Produzida', 'Qtd_Refugada', 'Qtd_Retrabalhada', 'Consumo total de Massa Acumulada', 'Cod Produto']]
y = df['Tempo Restante para Manutenção']

# Divida os dados em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crie uma lista de modelos a serem avaliados
models = {
    "SVR": SVR(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Decision Tree": DecisionTreeRegressor()
}

# Itere sobre os modelos, treine e avalie o desempenho
results = []
for model_name, model in models.items():
    # Treine o modelo
    model.fit(X_train, y_train)

    # Faça previsões no conjunto de teste
    y_pred = model.predict(X_test)

    # Calcule as métricas

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [8]:
# Split into features and target
X = df.drop("Tempo Restante para Manutenção", axis=1)
y = df["Tempo Restante para Manutenção"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and evaluate models
models = {
    "SVR": SVR(kernel="rbf"),
    "Gradient Boosting": GradientBoostingRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100),
    "Decision Tree": DecisionTreeRegressor(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name}:")
    print(f"  Mean Squared Error: {mse:.2f}")
    print(f"  R-squared: {r2:.2f}")

SVR:
  Mean Squared Error: 10939.53
  R-squared: 0.00
Gradient Boosting:
  Mean Squared Error: 11362.72
  R-squared: -0.04
Random Forest:
  Mean Squared Error: 11368.39
  R-squared: -0.04
Decision Tree:
  Mean Squared Error: 20728.55
  R-squared: -0.89


## Análise dos Resultados dos Modelos de Regressão

É importante substituir os dados da SABÓ para validar os modelos e predicoes. Com base nos resultados, podemos observar que o **SVR** tem o menor erro quadrático médio (MSE), mas o **R-quadrado** está próximo de zero para todos os modelos, indicando que eles **não conseguem explicar bem a variabilidade nos dados**.
